In [0]:
%pip install lightgbm optuna shap
dbutils.library.restartPython()

In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Feature Engineering — Loan Default Prediction
# MAGIC Builds a point-in-time-correct loan-month observation table:
# MAGIC one row per (loan_account_id, month), features known ONLY as of that
# MAGIC month, label = does this loan hit DEFAULT/NPA/WRITTEN_OFF within the
# MAGIC next 3 loan-months.

# COMMAND ----------

catalog = "credit_risk_fraud_detection"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.ml")

# COMMAND ----------

spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.ml.loan_month_features AS

WITH loan_static AS (
    select distinct
        loan_account_id,
        customer_id,
        loan_type,
        interest_rate,
        tenure_months,
        emi_amount,
        disbursed_amount,
        cibil_score_at_decision
    from {catalog}.silver.dim_loan_account
),

persona_lookup as (
    select customer_id, persona
    from {catalog}.silver.customer_snapshot
    where dbt_valid_to is null
),

-- one row per loan-month, already point-in-time by construction
events as (
    select
        loan_account_id,
        customer_id,
        installment_number,
        due_date,
        dpd_at_event,
        loan_status_at_event,
        consecutive_missed_at_event,
        is_missed,
        is_partial,
        salary_credited_this_month
    from {catalog}.silver.fct_repayment_events
),

-- rolling 3/6-month missed & partial payment counts, using ONLY prior rows
rolling as (
    select
        *,
        sum(case when is_missed then 1 else 0 end) over (
            partition by loan_account_id order by installment_number
            rows between 3 preceding and 1 preceding
        ) as missed_last_3m,
        sum(case when is_missed then 1 else 0 end) over (
            partition by loan_account_id order by installment_number
            rows between 6 preceding and 1 preceding
        ) as missed_last_6m,
        sum(case when is_partial then 1 else 0 end) over (
            partition by loan_account_id order by installment_number
            rows between 3 preceding and 1 preceding
        ) as partial_last_3m
    from events
),

-- NACH bounce count as of each loan-month (bounces up to and including this month)
bounce_counts as (
    select
        loan_account_id,
        nach_debit_date,
        consecutive_bounce_count
    from {catalog}.silver.fct_nach_bounces
),

with_bounces as (
    select
        r.*,
        (
            select max(b.consecutive_bounce_count)
            from bounce_counts b
            where b.loan_account_id = r.loan_account_id
              and b.nach_debit_date <= r.due_date
        ) as bounce_count_as_of_month
    from rolling r
),

-- forward-looking label: does the loan hit a bad status in the NEXT 3 loan-months
labeled as (
    select
        w.*,
        max(case
            when f.loan_status_at_event in ('60_DPD', '90_DPD', 'DEFAULT', 'NPA')
            then 1 else 0
        end) as will_escalate_next_3m,
        count(f.installment_number) as future_months_available
    from with_bounces w
    left join events f
        on w.loan_account_id = f.loan_account_id
        and f.installment_number between w.installment_number + 1 and w.installment_number + 3
    group by
        w.loan_account_id, w.customer_id, w.installment_number, w.due_date,
        w.dpd_at_event, w.loan_status_at_event, w.consecutive_missed_at_event,
        w.is_missed, w.is_partial, w.salary_credited_this_month,
        w.missed_last_3m, w.missed_last_6m, w.partial_last_3m, w.bounce_count_as_of_month
)

select
    l.loan_account_id,
    l.customer_id,
    ls.loan_type,
    ls.interest_rate,
    ls.tenure_months,
    ls.emi_amount,
    ls.disbursed_amount,
    ls.cibil_score_at_decision,
    p.persona,
    l.installment_number,
    l.due_date,
    l.dpd_at_event,
    l.loan_status_at_event,
    l.consecutive_missed_at_event,
    l.salary_credited_this_month,
    coalesce(l.missed_last_3m, 0)      as missed_last_3m,
    coalesce(l.missed_last_6m, 0)      as missed_last_6m,
    coalesce(l.partial_last_3m, 0)     as partial_last_3m,
    coalesce(l.bounce_count_as_of_month, 0) as bounce_count_as_of_month,
    l.will_escalate_next_3m
from labeled l
join loan_static ls on l.loan_account_id = ls.loan_account_id
left join persona_lookup p on l.customer_id = p.customer_id
where
    -- observation population restricted to loans NOT already at 60_DPD+ --
    -- we're predicting forward escalation risk, not re-identifying loans
    -- that are already known-bad
    l.loan_status_at_event in ('ACTIVE', 'CURRENT', '30_DPD')
    and l.future_months_available = 3
""")

print(spark.table(f"{catalog}.ml.loan_month_features").count(), "loan-month observations")

# COMMAND ----------

display(
    spark.sql(f"""
        select will_escalate_next_3m, count(*) as n
        from {catalog}.ml.loan_month_features
        group by will_escalate_next_3m
    """)
)

In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Model Training — Delinquency Escalation Predictor (LightGBM)

# COMMAND ----------

import mlflow
import lightgbm as lgb
import optuna
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve
import shap
import matplotlib.pyplot as plt
from mlflow import MlflowClient
from mlflow.models import infer_signature

catalog = "credit_risk_fraud_detection"
mlflow.set_registry_uri("databricks-uc")

# COMMAND ----------

df = spark.table(f"{catalog}.ml.loan_month_features").toPandas()
print(df.shape)
print(df["will_escalate_next_3m"].value_counts())

# COMMAND ----------

# MAGIC %md ## Feature prep

# COMMAND ----------

categorical_cols = ["loan_type", "persona", "loan_status_at_event"]
for c in categorical_cols:
    df[c] = df[c].astype("category")

feature_cols = [
    "interest_rate", "tenure_months", "emi_amount", "disbursed_amount",
    "cibil_score_at_decision", "dpd_at_event", "consecutive_missed_at_event",
    "missed_last_3m", "missed_last_6m", "partial_last_3m",
    "bounce_count_as_of_month", "salary_credited_this_month",
] + categorical_cols

target_col = "will_escalate_next_3m"

# COMMAND ----------

# MAGIC %md
# MAGIC ## Temporal + grouped split
# MAGIC Split by loan cohort (earliest due_date per loan), not by row --
# MAGIC every month of a given loan stays entirely on one side, avoiding both
# MAGIC temporal leakage and loan-identity leakage.

# COMMAND ----------

loan_first_seen = df.groupby("loan_account_id")["due_date"].min().sort_values()
loans_sorted = loan_first_seen.index.tolist()
n = len(loans_sorted)

train_loans = set(loans_sorted[: int(n * 0.70)])
val_loans   = set(loans_sorted[int(n * 0.70): int(n * 0.85)])
test_loans  = set(loans_sorted[int(n * 0.85):])

train_df = df[df["loan_account_id"].isin(train_loans)]
val_df   = df[df["loan_account_id"].isin(val_loans)]
test_df  = df[df["loan_account_id"].isin(test_loans)]

for name, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"{name}: {len(d)} rows, {d[target_col].sum()} positive")

X_train, y_train = train_df[feature_cols], train_df[target_col]
X_val, y_val = val_df[feature_cols], val_df[target_col]
X_test, y_test = test_df[feature_cols], test_df[target_col]

# COMMAND ----------

# MAGIC %md ## Optuna hyperparameter search (50 trials)

# COMMAND ----------

def ks_statistic(y_true, y_score):
    fpr, tpr, _ = roc_curve(y_true, y_score)
    return max(tpr - fpr)

def objective(trial):
    params = {
        "objective": "binary",
        "metric": "auc",
        "is_unbalance": True,
        "num_leaves": trial.suggest_int("num_leaves", 8, 64),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 100, 500),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 50),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
    }
    model = lgb.LGBMClassifier(**params)
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        categorical_feature=categorical_cols,
        callbacks=[lgb.early_stopping(30, verbose=False)],
    )
    preds = model.predict_proba(X_val)[:, 1]
    return roc_auc_score(y_val, preds)

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)
print("Best val AUC:", study.best_value)
print("Best params:", study.best_params)

# COMMAND ----------

# MAGIC %md ## Train final model, log to MLflow, register in Unity Catalog

# COMMAND ----------

with mlflow.start_run(run_name="lightgbm_delinquency_escalation") as run:
    best_params = dict(study.best_params)
    best_params.update({"objective": "binary", "metric": "auc", "is_unbalance": True})

    final_model = lgb.LGBMClassifier(**best_params)
    final_model.fit(
        pd.concat([X_train, X_val]), pd.concat([y_train, y_val]),
        categorical_feature=categorical_cols,
    )

    test_preds = final_model.predict_proba(X_test)[:, 1]
    test_auc = roc_auc_score(y_test, test_preds)
    test_ks = ks_statistic(y_test, test_preds)
    test_pr_auc = average_precision_score(y_test, test_preds)

    mlflow.log_params(best_params)
    mlflow.log_metric("test_roc_auc", test_auc)
    mlflow.log_metric("test_ks_statistic", test_ks)
    mlflow.log_metric("test_pr_auc", test_pr_auc)

    print(f"Test ROC-AUC: {test_auc:.4f}")
    print(f"Test KS:      {test_ks:.4f}")
    print(f"Test PR-AUC:  {test_pr_auc:.4f}")

    # SHAP summary plot
    explainer = shap.TreeExplainer(final_model)
    shap_values = explainer.shap_values(X_test)
    plot_values = shap_values[1] if isinstance(shap_values, list) else shap_values
    shap.summary_plot(plot_values, X_test, show=False)
    plt.savefig("/tmp/shap_summary.png", bbox_inches="tight")
    mlflow.log_artifact("/tmp/shap_summary.png")
    plt.close()

    # PR curve
    from sklearn.metrics import PrecisionRecallDisplay
    PrecisionRecallDisplay.from_predictions(y_test, test_preds)
    plt.savefig("/tmp/pr_curve.png", bbox_inches="tight")
    mlflow.log_artifact("/tmp/pr_curve.png")
    plt.close()

    signature = infer_signature(X_test, final_model.predict(X_test))

    mlflow.lightgbm.log_model(
        final_model,
        artifact_path="model",
        registered_model_name=f"{catalog}.ml.delinquency_escalation_model",
        signature=signature,
        input_example=X_test.head(5),
    )

    run_id = run.info.run_id

print("Run ID:", run_id)

# COMMAND ----------

# MAGIC %md ## Set @champion alias

# COMMAND ----------

client = MlflowClient()
model_name = f"{catalog}.ml.delinquency_escalation_model"
versions = client.search_model_versions(f"name='{model_name}'")
latest_version = max(int(v.version) for v in versions)

client.set_registered_model_alias(model_name, "champion", latest_version)
print(f"Set @champion alias -> version {latest_version}")